# Phase 26 — Full-system integration

Reconstructed notebook for `phase_26_full_four_domain_training`. Phase 26 bound the four components into one deployable bundle and verified the state machine. It did **not** jointly train the assembly.

> **Status:** reconstructed analysis notebook. It reads persisted artifacts only. No training, no new inference, no protected TEST access.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


In [3]:
inv = pd.read_csv(TABLES / "REMEDFILM_COMPONENT_PARAMETERS.csv")
inv[["symbol", "model_class", "checkpoint", "role", "conditional", "parameters"]].to_string(index=False)

' symbol           model_class                                                            checkpoint             role  conditional  parameters\n      G         GlobalMedFiLM checkpoints/phase21_global_medfilm_fourdomain_seed42_clean_v3/best.pt GLOBAL_SEGMENTER        False     3523001\n  R_psi BARRRegionalRefinerV1 checkpoints/phase25a_barr_regional_refiner_v1_1_roi030/final/final.pt REGIONAL_REFINER         True     1276401\n  B_phi   BARRBenefitRouterV1 checkpoints/phase25d_barr_benefit_router_v1_2_1_seed42/final/final.pt   BENEFIT_ROUTER         True       36801\nS_omega      BARRPostSafetyV1      checkpoints/phase25f_barr_post_safety_v1_0_seed42/final/final.pt      POST_SAFETY         True        1217\n   FULL             assembled                              bundle of the four component checkpoints  PROPOSED_SYSTEM         True     4837420'

In [4]:
tests = pd.read_csv(REPO / "reports/tables/phase26_integration_test_results.csv")
print("integration tests:", len(tests), "| all pass:",
      bool(tests["pass"].astype(str).str.lower().eq("true").all()))
tests

integration tests: 7 | all pass: True


,test,pass,detail
0,global_medfilm_io,True,"shape=(1, 1, 512, 512)"
1,router_feature_width_202,True,NaN
2,benefit_router_218d_input,True,"shape=(1,)"
3,regional_refiner_4ch_256,True,"shape=(1, 1, 256, 256)"
4,post_safety_20d_input,True,"shape=(1,)"
5,fusion_shape_compatibility,True,"shape=(8, 8)"
6,fusion_outside_roi_unmodified,True,NaN


In [5]:
sm = pd.read_csv(REPO / "reports/tables/phase26_state_machine_test_results.csv")
print("state-machine tests:", len(sm), "| all pass:",
      bool(sm["pass"].astype(str).str.lower().eq("true").all()))
sm

state-machine tests: 4 | all pass: True


,case,expected_state,actual_state,global,refiner,s_omega,pass
0,A_action_unavailable,FORCED_SKIP,FORCED_SKIP,1,0,0,True
1,B_router_skip,SKIP_GLOBAL,SKIP_GLOBAL,1,0,0,True
2,C_execute_accept,EXECUTE_THEN_ACCEPT,EXECUTE_THEN_ACCEPT,1,1,1,True
3,D_execute_reject,EXECUTE_THEN_REJECT,EXECUTE_THEN_REJECT,1,1,1,True


In [6]:
# Bundle integrity: every component checkpoint hash is bound in the manifest.
bm = json.loads((REPO / "reports/audits/phase26_bundle_manifest_integrity.json").read_text(encoding="utf-8"))
for k in list(bm)[:10]:
    v = bm[k]
    if not isinstance(v, (dict, list)):
        print(f"  {k}: {v}")

  artifact: phase26_bundle_manifest_integrity
  phase: 26
  method: RA-MedFiLM-BARR
  bundle_revision: ra_medfilm_barr_bundle_v1_0
  all_pass: True
  terminal_on_failure: PHASE26_BLOCKED__CANONICAL_BUNDLE_INTEGRITY_FAILURE


In [7]:
print("Phase 26 establishes the FULL system identity:")
print("  ReMedFiLM FULL = G + R_psi + B_phi + S_omega, assembled for inference.")
print("  The bundle is a deployment artifact; it is not a new trained model.")

Phase 26 establishes the FULL system identity:
  ReMedFiLM FULL = G + R_psi + B_phi + S_omega, assembled for inference.
  The bundle is a deployment artifact; it is not a new trained model.


## Provenance

- `reports/tables/phase26_component_inventory.csv`
- `reports/tables/phase26_integration_test_results.csv`
- `reports/audits/phase26_bundle_manifest_integrity.json`

In [8]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
